<a href="https://colab.research.google.com/github/dayeon-yu/income-prediction-catboost/blob/main/notebooks/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 소득 예측 — 추가 분석 노트북 (EDA · 모델 비교 · 검증 · 해석 · 공정성)

> `code.ipynb`가 **제출용 파이프라인(CatBoost, 최종 0.8339)** 이라면, 이 노트북은 과제 안내문이 요구한
> "여러 모델 비교 · 모델 해석 · 오류 분석 · 공정성 분석"을 **같은 피처(31개)** 위에서 수행하고,
> 제출 파이프라인의 검증 방식이 얼마나 낙관적인지 점검한다.
>
> **모델 선택 규칙**: `catboost`가 설치돼 있으면 CatBoost(분석용 경량 설정)를, 없으면 scikit-learn의
> `HistGradientBoostingClassifier`를 같은 역할로 사용한다. 아래 출력은 **설치 환경에 따라** 둘 중 하나로 표시된다.
> 모든 난수 seed = 42.

**목차**
1. 환경·데이터 로드
2. EDA — 클래스 불균형, 결측, 수치형/범주형 변수와 소득의 관계
3. 모델 비교 — Logistic Regression · Random Forest · Gradient Boosting(+ CatBoost 제출 결과)
4. 피처 블록 ablation — Base FE / Target Encoding / Group Aggregation의 기여
5. 검증 방식 점검 — Target Encoding 시점에 따른 누수, 5-fold CV, threshold 안정성
6. 모델 해석 — permutation importance
7. 오류 분석 — 어떤 집단에서 틀리는가
8. 공정성 분석 — 성별·인종별 예측 양성률 / TPR / FPR
9. 결론

In [ ]:
   !git clone https://github.com/dayeon-yu/income-prediction-catboost.git
   %cd /content/income-prediction-catboost/notebooks
   !pip install catboost -q

fatal: destination path 'income-prediction-catboost' already exists and is not an empty directory.
/content/income-prediction-catboost/notebooks


## 1. 환경 · 데이터 로드

교수님이 배포한 `train.csv`(39,073 × 15) / `test.csv`(9,769 × 14)를 그대로 사용한다. 전처리·피처 함수는 `src/features.py`에 있고 `code.ipynb`와 동일한 로직이다.

In [ ]:
import sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")   # 저장소 루트의 src/ 를 import

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from src.features import load_data, clean, build_features, build_features_split_first, CAT_COLS
from src.evaluate import project_metric, best_threshold, group_fairness
from src.viz import setup, save_show, C_LOW, C_HIGH, C_NEUTRAL, C_ACCENT
setup()

RANDOM_STATE = 42
TRAIN_PATH, TEST_PATH = "../data/train.csv", "../data/test.csv"
train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)
print("train:", train_df.shape, "| test:", test_df.shape)

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
print("CatBoost available:", HAS_CATBOOST, "→ 분석용 GBDT =", "CatBoost(iterations=1000, lr=0.05)" if HAS_CATBOOST else "HistGradientBoosting(max_iter=600, lr=0.05)")

train: (39073, 15) | test: (9769, 14)
CatBoost available: True → 분석용 GBDT = CatBoost(iterations=1000, lr=0.05)


## 2. EDA

### 2.1 클래스 분포와 결측

`>50K`는 23.9%로 약 3:1 불균형이다. 결측은 `workclass`·`occupation`·`native_country` 세 변수에만 있고, `workclass` 결측 2,213건은 **전부 `occupation`도 결측**이다(동시 결측). 결측 집단의 고소득 비율은 9.1%로 전체(24.8%)보다 훨씬 낮아, 결측을 대체하지 않고 `Unknown` 카테고리로 **보존**하는 편이 정보를 지킨다.

In [ ]:
df = clean(train_df)
df["y"] = (df["income"].str.strip() == ">50K").astype(int)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
vc = df["income"].value_counts()
ax[0].bar(vc.index, vc.values, color=[C_LOW, C_HIGH], width=0.55)
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom", fontsize=9)
ax[0].set_title("Class distribution (train, n=39,073)"); ax[0].set_ylim(0, vc.max() * 1.25); ax[0].set_ylabel("rows")

miss = train_df.isna().mean().sort_values(ascending=False); miss = miss[miss > 0] * 100
ax[1].barh(miss.index[::-1], miss.values[::-1], color=C_NEUTRAL, height=0.5)
for i, (k, v) in enumerate(list(miss.items())[::-1]):
    ax[1].text(v + 0.1, i, f"{v:.2f}%", va="center", fontsize=9)
ax[1].set_title('Missing values (%) — kept as "Unknown"'); ax[1].set_xlim(0, miss.max() * 1.3)
save_show("01_class_missing")

m = train_df[["workclass", "occupation"]].isna()
print(f"workclass·occupation 동시 결측: {(m.workclass & m.occupation).sum():,}건 / occupation만 결측: {(~m.workclass & m.occupation).sum()}건")
print(f">50K 비율 — workclass 결측 집단 {df.loc[train_df.workclass.isna(), 'y'].mean():.1%} vs 정상 {df.loc[~train_df.workclass.isna(), 'y'].mean():.1%}")
print(f"완전 중복 행(id 제외): {train_df.drop(columns='id').duplicated().sum():,}건 ({train_df.drop(columns='id').duplicated().mean():.1%})")

workclass·occupation 동시 결측: 2,213건 / occupation만 결측: 10건
>50K 비율 — workclass 결측 집단 9.1% vs 정상 24.8%
완전 중복 행(id 제외): 4,482건 (11.5%)


### 2.2 수치형 변수

`education_num`·`age`·`hours_per_week`는 분포가 오른쪽으로 이동하고, `capital_gain`은 대부분 0이지만 **5,000 이상이면 고소득 비율이 82%→99%로 급증**하며 99,999는 206명 전원이 고소득이다. 선형 관계라기보다 임계값을 넘으면 클래스가 갈리는 **규칙형 신호**라서 트리 기반 모델에 유리하다.

In [ ]:
num = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for a, c in zip(axes, num):
    x0, x1 = df.loc[df.y == 0, c], df.loc[df.y == 1, c]
    if c.startswith("capital"):
        x0, x1 = np.log1p(x0), np.log1p(x1); a.set_xlabel(f"log1p({c})")
    else:
        a.set_xlabel(c)
    a.hist(x0, bins=30, density=True, alpha=0.75, color=C_LOW, label="<=50K")
    a.hist(x1, bins=30, density=True, alpha=0.65, color=C_HIGH, label=">50K")
    a.set_yticks([])
axes[0].legend(loc="upper right"); axes[0].set_title("Numeric features by income class", loc="left")
save_show("02_numeric_by_class")

cg = df.groupby(pd.cut(df.capital_gain, [-1, 0, 5000, 10000, 20000, 99998, 99999]), observed=True)["y"].agg(["mean", "size"])
cg.index = ["0", "1–5k", "5k–10k", "10k–20k", "20k–99,998", "99,999"]
display(pd.DataFrame({"n": cg["size"], ">50K rate (%)": (cg["mean"] * 100).round(1)}))

corr = df[num + ["y"]].corr()["y"].drop("y").sort_values(ascending=False)
print("Pearson corr with target:", corr.round(3).to_dict())

,n,>50K rate (%)
0,35866,20.5
1–5k,1246,18.7
5k–10k,1048,82.2
10k–20k,595,99.0
"20k–99,998",112,92.0
"99,999",206,100.0


Pearson corr with target: {'education_num': 0.335, 'age': 0.23, 'hours_per_week': 0.227, 'capital_gain': 0.223, 'capital_loss': 0.145}


### 2.3 범주형 변수

혼인상태·직업·학력·가구관계는 범주 간 고소득 비율 차이가 매우 크다(`Married-civ-spouse` 44.4% vs `Never-married` 4.6%, `Prof-school` 74.5% vs `HS-grad` 15.8%).

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for a, c in zip(axes, ["marital_status", "occupation", "education", "relationship"]):
    g = df.groupby(c)["y"].agg(["mean", "size"]).sort_values("mean")
    a.barh(g.index, g["mean"] * 100, color=C_HIGH, height=0.6, alpha=0.9)
    for i, (mm, n) in enumerate(zip(g["mean"], g["size"])):
        a.text(mm * 100 + 0.8, i, f"{mm*100:.1f}%  (n={n:,})", va="center", fontsize=7.5, color="#333")
    a.axvline(df.y.mean() * 100, color=C_NEUTRAL, ls="--", lw=1); a.set_xlim(0, 100)
    a.set_title(f">50K rate by {c}"); a.tick_params(axis="y", labelsize=8)
fig.suptitle("Dashed line = overall >50K rate 23.9%", y=1.02, fontsize=9, color="#555")
save_show("03_rate_by_category")

### 2.4 성별 격차는 어디서 오는가 — 상호작용

전체 고소득 비율은 남성 30.4% vs 여성 10.9%로 차이가 크지만, **같은 혼인상태 안에서 비교하면 `Married-civ-spouse` 집단은 남녀가 44~45%로 거의 같다.** 성별 격차의 상당 부분이 혼인상태 구성비 차이(남성 표본의 기혼 비율이 훨씬 높음)에서 온다는 뜻이다. 이 관찰은 8장 공정성 분석의 해석 근거가 되고, `sex_marital` 상호작용 피처를 만든 이유이기도 하다.

In [ ]:
piv = df.pivot_table(index="marital_status", columns="sex", values="y", aggfunc="mean").sort_values("Male") * 100
cnt = df.pivot_table(index="marital_status", columns="sex", values="y", aggfunc="size")
fig, ax = plt.subplots(figsize=(7, 3.6))
yy = np.arange(len(piv)); h = 0.38
ax.barh(yy - h/2, piv["Female"], h, color=C_LOW, label="Female")
ax.barh(yy + h/2, piv["Male"], h, color=C_HIGH, label="Male")
ax.set_yticks(yy); ax.set_yticklabels(piv.index, fontsize=8); ax.legend(); ax.set_xlim(0, 60)
ax.set_title(">50K rate (%) — sex × marital_status")
save_show("04_sex_marital")
share = df.groupby("sex")["marital_status"].value_counts(normalize=True).unstack().loc[:, ["Married-civ-spouse", "Never-married"]] * 100
print("혼인상태 구성비(%):"); display(share.round(1))

혼인상태 구성비(%):


marital_status,Married-civ-spouse,Never-married
sex,,
Female,15.2,44.3
Male,61.2,27.1


## 3. 모델 비교 — 같은 31개 피처, 같은 holdout

과제 안내문은 "여러 알고리즘을 비교"하라고 요구한다. `code.ipynb`와 **동일한 피처(31개)·동일한 stratified holdout(80/20, seed 42)** 에서 세 모델을 학습하고, 과제 지표 `(F1+AUC)/2`를 (a) threshold 0.5, (b) 검증셋에서 F1을 최대화하는 threshold 두 가지로 계산한다.

- Logistic Regression: 범주형 One-Hot(빈도 5 미만 묶음) + 수치형 표준화
- Random Forest: 500 trees, `min_samples_leaf=2`
- Gradient Boosting: CatBoost(설치 시) 또는 HistGradientBoosting(범주형 native 처리)
- **CatBoost 제출 모델(iterations 2500, lr 0.02, depth 7)** 은 `code.ipynb` 실행 결과(AUC 0.9306 / F1 0.7373 / 0.8339)를 참고 행으로 넣는다.

In [ ]:
X, y, X_test, test_ids, cat_features = build_features(train_df, test_df, random_state=RANDOM_STATE)
num_features = [c for c in X.columns if c not in cat_features]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print("features:", X.shape[1], "| train/val:", X_tr.shape[0], X_va.shape[0])

def make_lr():
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), cat_features),
                             ("num", StandardScaler(), num_features)])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, C=1.0))])

def make_rf():
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_features)],
                            remainder="passthrough")
    return Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=500, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE))])

class CatBoostWrapper:
    '''sklearn Pipeline 없이 DataFrame을 그대로 받는 간단한 래퍼 (분석용 경량 설정).'''
    def __init__(self, cats, iterations=1000, lr=0.05, depth=7):
        self.cats, self.params = cats, dict(iterations=iterations, learning_rate=lr, depth=depth, l2_leaf_reg=10,
                                            random_seed=RANDOM_STATE, verbose=False)
    def fit(self, Xf, yf):
        self.m = CatBoostClassifier(cat_features=self.cats, **self.params).fit(Xf, yf); return self
    def predict_proba(self, Xf): return self.m.predict_proba(Xf)

def make_gbdt(cats=None, nums=None):
    cats = cat_features if cats is None else cats
    nums = num_features if nums is None else nums
    if HAS_CATBOOST:
        return CatBoostWrapper(cats)
    pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cats)],
                            remainder="passthrough", verbose_feature_names_out=False)
    pre.set_output(transform="pandas")
    clf = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=600, max_leaf_nodes=31, l2_regularization=1.0,
                                         categorical_features=[True]*len(cats) + [False]*len(nums),
                                         early_stopping=False, random_state=RANDOM_STATE)
    return Pipeline([("pre", pre), ("clf", clf)])

GBDT_NAME = "CatBoost (분석용, it=1000)" if HAS_CATBOOST else "HistGradientBoosting"

def evaluate_model(model, X_tr, y_tr, X_va, y_va):
    t = time.time(); model.fit(X_tr, y_tr); p = model.predict_proba(X_va)[:, 1]
    th, _, _, _ = best_threshold(y_va.values, p)
    r05, rb = project_metric(y_va, p, 0.5), project_metric(y_va, p, th)
    return {"AUC": r05["AUC"], "F1@0.5": r05["F1"], "(F1+AUC)/2 @0.5": r05["(F1+AUC)/2"],
            "best threshold": th, "F1@best": rb["F1"], "(F1+AUC)/2 @best": rb["(F1+AUC)/2"], "fit sec": time.time() - t}, p

results, val_probs = {}, {}
for name, mk in [("Logistic Regression", make_lr), ("Random Forest", make_rf), (GBDT_NAME, make_gbdt)]:
    results[name], val_probs[name] = evaluate_model(mk(), X_tr, y_tr, X_va, y_va)
results["CatBoost 제출 모델 (code.ipynb, it=2500)"] = {"AUC": 0.9306, "F1@0.5": np.nan, "(F1+AUC)/2 @0.5": np.nan,
                                                   "best threshold": 0.422, "F1@best": 0.7373, "(F1+AUC)/2 @best": 0.8339, "fit sec": np.nan}
cmp_df = pd.DataFrame(results).T
display(cmp_df.round(4))
cmp_df.round(4).to_csv("../results/model_comparison.csv")

fig, ax = plt.subplots(figsize=(7.5, 3))
names = list(cmp_df.index); vals = cmp_df["(F1+AUC)/2 @best"].values
yy = np.arange(len(names))[::-1]
for yi, n, v in zip(yy, names, vals):
    ax.plot([0.80, v], [yi, yi], color="#ddd", lw=1.2, zorder=1)
    ax.scatter(v, yi, s=70, color=C_HIGH if "제출" in n else C_LOW, zorder=2)
    ax.text(v + 0.0012, yi, f"{v:.4f}", va="center", fontsize=9)
ax.set_yticks(yy); ax.set_yticklabels(names); ax.set_xlim(0.80, 0.845); ax.grid(axis="y", alpha=0)
ax.set_title("(F1+AUC)/2 at best threshold — same 31 features, same holdout")
save_show("05_model_comparison")

features: 31 | train/val: 31258 7815


,AUC,F1@0.5,(F1+AUC)/2 @0.5,best threshold,F1@best,(F1+AUC)/2 @best,fit sec
Logistic Regression,0.9159,0.6800,0.7979,0.380,0.7114,0.8136,11.0470
Random Forest,0.9167,0.6845,0.8006,0.343,0.7165,0.8166,31.9805
"CatBoost (분석용, it=1000)",0.9301,0.7209,0.8255,0.424,0.7342,0.8321,125.3532
"CatBoost 제출 모델 (code.ipynb, it=2500)",0.9306,NaN,NaN,0.422,0.7373,0.8339,NaN


## 4. 피처 블록 ablation — 무엇이 실제로 기여했나

보고서는 "CatBoost가 스스로 학습할 수 없는 cross-row 통계(Target Encoding, Group Aggregation)만 성능에 기여했다"고 주장한다.
이 주장을 **숫자로** 확인하기 위해 같은 holdout에서 피처 블록을 하나씩 추가하며 GBDT를 다시 학습한다.

In [ ]:
y_all = (train_df["income"].str.strip() == ">50K").astype(int)
idx_tr, idx_va = train_test_split(np.arange(len(train_df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y_all)

def eval_block(Xb, cats):
    nums = [c for c in Xb.columns if c not in cats]
    r, p = evaluate_model(make_gbdt(cats, nums), Xb.iloc[idx_tr], y_all.iloc[idx_tr], Xb.iloc[idx_va], y_all.iloc[idx_va])
    return {"n_features": Xb.shape[1], "AUC": r["AUC"], "F1@best": r["F1@best"], "(F1+AUC)/2": r["(F1+AUC)/2 @best"], "best th": r["best threshold"]}

abl = {}
Xraw = clean(train_df.drop(columns=["income", "id"]))
abl["원본 13개 (클리닝만)"] = eval_block(Xraw, CAT_COLS)
for te_flag, ga_flag, label in [(False, False, "+ Base FE 6개 → 19"), (True, False, "+ Target Encoding 5개 → 24"), (True, True, "+ Group Aggregation 7개 → 31 (최종)")]:
    Xb, _, _, _, cats_b = build_features(train_df, test_df, target_encoding=te_flag, group_agg=ga_flag, random_state=RANDOM_STATE)
    abl[label] = eval_block(Xb, cats_b)
abl_df = pd.DataFrame(abl).T
abl_df["Δ vs 원본"] = abl_df["(F1+AUC)/2"] - abl_df.loc["원본 13개 (클리닝만)", "(F1+AUC)/2"]
display(abl_df.round(4))
abl_df.round(4).to_csv("../results/ablation_feature_blocks.csv")

,n_features,AUC,F1@best,(F1+AUC)/2,best th,Δ vs 원본
원본 13개 (클리닝만),13.0,0.9304,0.7380,0.8342,0.421,0.0000
+ Base FE 6개 → 19,19.0,0.9300,0.7343,0.8321,0.409,-0.0021
+ Target Encoding 5개 → 24,24.0,0.9301,0.7336,0.8318,0.434,-0.0024
+ Group Aggregation 7개 → 31 (최종),31.0,0.9301,0.7342,0.8321,0.424,-0.0021


## 5. 검증 방식 점검

### 5.1 Target Encoding을 "split 전에" 계산하면 생기는 누수
`code.ipynb`는 전체 train(39,073행)에 대해 OOF Target Encoding과 Group Aggregation을 계산한 **다음** train/val을 나눈다.
OOF라서 각 행이 자기 자신의 타겟을 보지는 않지만, **train 행의 인코딩 값에 val 행의 타겟이 섞여 들어간다.**
아래는 (a) 현재 방식과 (b) split을 먼저 한 뒤 train 통계로만 val을 인코딩하는 방식의 차이다.

In [ ]:
Xa, ya, Xb, yb, cats_sf = build_features_split_first(train_df.iloc[idx_tr], train_df.iloc[idx_va], random_state=RANDOM_STATE)
nums_sf = [c for c in Xa.columns if c not in cats_sf]
r_cur = abl_df.loc["+ Group Aggregation 7개 → 31 (최종)"]
r_sf, p_sf = evaluate_model(make_gbdt(cats_sf, nums_sf), Xa, ya, Xb, yb)
leak_df = pd.DataFrame({
    "(a) 전체 train에서 인코딩 후 split — 현재 code.ipynb": {"AUC": r_cur["AUC"], "F1@best": r_cur["F1@best"], "(F1+AUC)/2": r_cur["(F1+AUC)/2"]},
    "(b) split 먼저 → train 통계로만 val 인코딩": {"AUC": r_sf["AUC"], "F1@best": r_sf["F1@best"], "(F1+AUC)/2": r_sf["(F1+AUC)/2 @best"]},
}).T
leak_df["Δ (a)−(b)"] = leak_df["(F1+AUC)/2"] - leak_df["(F1+AUC)/2"].iloc[1]
display(leak_df.round(4))
leak_df.round(4).to_csv("../results/te_leakage_check.csv")

,AUC,F1@best,(F1+AUC)/2,Δ (a)−(b)
(a) 전체 train에서 인코딩 후 split — 현재 code.ipynb,0.9301,0.7342,0.8321,-0.0004
(b) split 먼저 → train 통계로만 val 인코딩,0.9299,0.7351,0.8325,0.0000


### 5.2 단일 holdout은 얼마나 믿을 수 있나 — 5-fold CV
holdout 하나로 고른 숫자(0.8339)와 threshold(0.422)가 분할에 따라 얼마나 흔들리는지, fold마다 split-first 인코딩을 다시 하는 5-fold CV로 확인한다.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(train_df)); folds = []
for k, (a, b) in enumerate(skf.split(train_df, y_all)):
    Xa, ya, Xb, yb, cats_k = build_features_split_first(train_df.iloc[a], train_df.iloc[b], random_state=RANDOM_STATE)
    nums_k = [c for c in Xa.columns if c not in cats_k]
    r, p = evaluate_model(make_gbdt(cats_k, nums_k), Xa, ya, Xb, yb); oof[b] = p
    folds.append({"fold": k, "AUC": r["AUC"], "F1@best": r["F1@best"], "(F1+AUC)/2": r["(F1+AUC)/2 @best"], "best th": r["best threshold"]})
cv_df = pd.DataFrame(folds).set_index("fold")
cv_df.loc["mean"] = cv_df.mean(); cv_df.loc["std"] = cv_df.iloc[:-1].std()
display(cv_df.round(4))
cv_df.round(4).to_csv("../results/cv_5fold.csv")

auc_oof = roc_auc_score(y_all, oof)
th_oof, f1_oof, grid, f1s = best_threshold(y_all.values, oof)
print(f"OOF 전체: AUC {auc_oof:.4f} | best threshold {th_oof:.3f} | F1 {f1_oof:.4f} | (F1+AUC)/2 {(f1_oof+auc_oof)/2:.4f}")
np.save("../results/oof_probs.npy", oof)

,AUC,F1@best,(F1+AUC)/2,best th
fold,,,,
0,0.9315,0.7333,0.8324,0.3950
1,0.9230,0.7247,0.8239,0.3820
2,0.9279,0.7282,0.8280,0.3960
3,0.9251,0.7228,0.8240,0.4740
4,0.9335,0.7410,0.8372,0.3760
mean,0.9282,0.7300,0.8291,0.4046
std,0.0043,0.0073,0.0057,0.0397


OOF 전체: AUC 0.9282 | best threshold 0.382 | F1 0.7284 | (F1+AUC)/2 0.8283


### 5.3 threshold 민감도
평가식에서 AUC는 threshold와 무관하므로 threshold 최적화 = F1 최대화다. OOF 예측으로 그린 F1 곡선은 **0.30~0.45 구간이 평평**해서, 그 안에서는 어떤 값을 골라도 F1 차이가 0.01 이내다. 즉 0.422라는 숫자 자체보다 "0.5 대신 0.35~0.45 어딘가"를 쓰는 결정이 중요하다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(grid, f1s, color=C_LOW, lw=2, label="F1 (OOF)")
ax.plot(grid, (f1s + auc_oof) / 2, color=C_HIGH, lw=2, label="(F1+AUC)/2 (OOF)")
ax.axvline(0.5, color=C_NEUTRAL, ls=":", lw=1); ax.text(0.505, f1s.min() + 0.01, "default 0.5", fontsize=8, color="#555")
ax.axvline(th_oof, color=C_ACCENT, ls="--", lw=1); ax.text(th_oof + 0.005, f1s.min() + 0.05, f"OOF best {th_oof:.3f}", fontsize=8, color=C_ACCENT)
for t in cv_df.loc[[0,1,2,3,4], "best th"]:
    ax.axvline(t, color=C_ACCENT, alpha=0.25, lw=1)
ax.set_xlabel("threshold"); ax.set_ylabel("score"); ax.legend(loc="lower left")
ax.set_title("Metric vs threshold (thin green lines = per-fold best thresholds)")
save_show("06_threshold_curve")
flat = grid[f1s >= f1s.max() - 0.01]
print(f"F1이 최대값−0.01 이내인 threshold 범위: {flat.min():.3f} ~ {flat.max():.3f}")

F1이 최대값−0.01 이내인 threshold 범위: 0.270 ~ 0.472


## 6. 모델 해석 — permutation importance
holdout(val)에서 각 피처를 무작위로 섞었을 때 AUC가 얼마나 떨어지는지로 중요도를 잰다(모델 종류와 무관하게 쓸 수 있는 방법).

In [ ]:
gb = make_gbdt(); gb.fit(X_tr, y_tr)
base_auc = roc_auc_score(y_va, gb.predict_proba(X_va)[:, 1])

def permutation_importance_auc(model, Xv, yv, n_repeats=3, seed=RANDOM_STATE):
    # 모델 종류와 무관한 permutation importance: 피처 하나를 섞었을 때의 AUC 하락량
    rng = np.random.default_rng(seed); rows = []
    for col in Xv.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = Xv.copy(); Xp[col] = rng.permutation(Xp[col].values)
            drops.append(base_auc - roc_auc_score(yv, model.predict_proba(Xp)[:, 1]))
        rows.append({"feature": col, "ΔAUC mean": np.mean(drops), "ΔAUC std": np.std(drops)})
    return pd.DataFrame(rows).sort_values("ΔAUC mean", ascending=False)

imp = permutation_importance_auc(gb, X_va, y_va)
print(f"baseline val AUC = {base_auc:.4f}")
display(imp.head(15).round(4).reset_index(drop=True))
imp.round(5).to_csv("../results/permutation_importance.csv", index=False)

top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top["feature"], top["ΔAUC mean"], xerr=top["ΔAUC std"], color=C_LOW, height=0.6, error_kw=dict(lw=0.8, color="#555"))
ax.set_xlabel("AUC drop when permuted (val)"); ax.set_title(f"Permutation importance — top 15 ({GBDT_NAME})")
save_show("07_permutation_importance")

baseline val AUC = 0.9301


,feature,ΔAUC mean,ΔAUC std
0,cap_net,0.0145,0.0004
1,relationship,0.0143,0.0005
2,hours_per_week,0.0122,0.0009
3,capital_gain,0.0115,0.0006
4,education,0.0080,0.0004
5,age_bins,0.0062,0.0004
6,age,0.0057,0.0003
7,capital_loss,0.0048,0.0004
8,sex_marital_te,0.0040,0.0002
9,sex_marital,0.0028,0.0002


## 7. 오류 분석 — 어떤 집단에서 틀리는가
OOF 예측(5-fold)과 OOF 최적 threshold로 전체 train의 혼동행렬을 만들고, False Negative(고소득인데 저소득으로 예측)·False Positive 비율이 높은 집단을 찾는다.

In [ ]:
pred = (oof >= th_oof).astype(int)
tn, fp, fn, tp = confusion_matrix(y_all, pred).ravel()
print(f"threshold={th_oof:.3f} | TN {tn:,}  FP {fp:,}  FN {fn:,}  TP {tp:,}")
print(f"precision {tp/(tp+fp):.3f} | recall {tp/(tp+fn):.3f} | F1 {f1_score(y_all, pred):.3f}")

err = clean(train_df).assign(y=y_all.values, prob=oof, pred=pred)
err["FN"] = ((err.y == 1) & (err.pred == 0)).astype(int)
err["FP"] = ((err.y == 0) & (err.pred == 1)).astype(int)
err["age_bin"] = pd.cut(err.age, [0, 25, 35, 45, 60, 100], labels=["≤25", "26–35", "36–45", "46–60", "60+"])

def err_table(col, min_n=300):
    g = err.groupby(col, observed=True).agg(n=("y", "size"), pos_rate=("y", "mean"),
                                          FN_rate=("FN", lambda s: s.sum() / max(1, (err.loc[s.index, "y"] == 1).sum())),
                                          FP_rate=("FP", lambda s: s.sum() / max(1, (err.loc[s.index, "y"] == 0).sum())))
    return g[g.n >= min_n].sort_values("FN_rate", ascending=False)

for col in ["education", "marital_status", "age_bin"]:
    print(f"\n[{col}] FN_rate = 고소득 중 놓친 비율, FP_rate = 저소득 중 잘못 양성 예측한 비율")
    display(err_table(col).round(3))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(err.loc[err.y == 0, "prob"], bins=50, alpha=0.75, color=C_LOW, label="<=50K (actual)")
ax.hist(err.loc[err.y == 1, "prob"], bins=50, alpha=0.65, color=C_HIGH, label=">50K (actual)")
ax.axvline(th_oof, color=C_ACCENT, ls="--", lw=1.2, label=f"threshold {th_oof:.3f}")
ax.set_xlabel("predicted P(>50K) — OOF"); ax.set_yticks([]); ax.legend(); ax.set_title("Score distribution by actual class")
save_show("08_score_distribution")

threshold=0.382 | TN 26,726  FP 3,003  FN 2,272  TP 7,072
precision 0.702 | recall 0.757 | F1 0.728

[education] FN_rate = 고소득 중 놓친 비율, FP_rate = 저소득 중 잘못 양성 예측한 비율


,n,pos_rate,FN_rate,FP_rate
education,,,,
9th,611,0.051,0.839,0.009
7th-8th,771,0.065,0.800,0.007
10th,1108,0.063,0.786,0.006
5th-6th,406,0.049,0.750,0.008
12th,513,0.078,0.725,0.013
11th,1455,0.051,0.581,0.006
HS-grad,12646,0.158,0.456,0.061
Some-college,8744,0.190,0.266,0.105
Assoc-voc,1627,0.256,0.260,0.162



[marital_status] FN_rate = 고소득 중 놓친 비율, FP_rate = 저소득 중 잘못 양성 예측한 비율


,n,pos_rate,FN_rate,FP_rate
marital_status,,,,
Widowed,1221,0.083,0.594,0.009
Never-married,12809,0.046,0.548,0.006
Separated,1233,0.065,0.538,0.013
Divorced,5328,0.102,0.519,0.018
Married-spouse-absent,504,0.093,0.511,0.011
Married-civ-spouse,17950,0.444,0.193,0.282



[age_bin] FN_rate = 고소득 중 놓친 비율, FP_rate = 저소득 중 잘못 양성 예측한 비율


,n,pos_rate,FN_rate,FP_rate
age_bin,,,,
≤25,7707,0.018,0.688,0.001
26–35,10148,0.188,0.364,0.072
60+,2855,0.236,0.352,0.076
36–45,9569,0.345,0.193,0.178
46–60,8794,0.378,0.183,0.206


## 8. 공정성 분석 — 성별 · 인종
과제 안내문이 명시한 추가 분석이다. 그룹별 **예측 양성률 · TPR(recall) · FPR · AUC** 를 비교한다.
해석할 때는 2.4절의 관찰(성별 격차의 상당 부분이 혼인상태 구성 차이)을 함께 봐야 한다.

In [ ]:
fair_sex = group_fairness(y_all, oof, err["sex"], th_oof)
fair_race = group_fairness(y_all, oof, err["race"], th_oof)
print("[sex]"); display(fair_sex.round(3))
print("[race]"); display(fair_race.round(3))
pd.concat([fair_sex.assign(attr="sex"), fair_race.assign(attr="race")]).round(4).to_csv("../results/fairness_by_group.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, fdf, title in [(axes[0], fair_sex, "sex"), (axes[1], fair_race, "race")]:
    xx = np.arange(len(fdf)); w = 0.26
    ax.bar(xx - w, fdf["actual_pos_rate"], w, color=C_NEUTRAL, label="actual >50K rate")
    ax.bar(xx, fdf["TPR(recall)"], w, color=C_LOW, label="TPR (recall)")
    ax.bar(xx + w, fdf["FPR"], w, color=C_HIGH, label="FPR")
    short = {"Asian-Pac-Islander": "Asian-Pac-\nIslander", "Amer-Indian-Eskimo": "Amer-Indian-\nEskimo"}
    ax.set_xticks(xx); ax.set_xticklabels([f"{short.get(g, g)}\n(n={n:,})" for g, n in zip(fdf.group, fdf.n)], fontsize=8)
    ax.set_title(f"By {title} (threshold {th_oof:.3f})")
axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1)
save_show("09_fairness_sex_race")

# 같은 혼인상태 안에서의 성별 비교 (구성 효과 통제)
sub = err[err.marital_status == "Married-civ-spouse"]
print("[Married-civ-spouse 집단 내 성별 비교]"); display(group_fairness(sub.y, sub.prob, sub.sex, th_oof).round(3))

[sex]


,group,n,actual_pos_rate,pred_pos_rate,TPR(recall),FPR,precision,F1,AUC
0,Male,26121,0.304,0.335,0.772,0.144,0.700,0.734,0.910
1,Female,12952,0.109,0.103,0.674,0.033,0.714,0.693,0.948


[race]


,group,n,actual_pos_rate,pred_pos_rate,TPR(recall),FPR,precision,F1,AUC
0,White,33402,0.254,0.275,0.763,0.109,0.705,0.733,0.925
1,Black,3774,0.121,0.117,0.660,0.042,0.683,0.671,0.951
2,Asian-Pac-Islander,1215,0.267,0.300,0.751,0.136,0.668,0.707,0.904
3,Amer-Indian-Eskimo,372,0.132,0.148,0.633,0.074,0.564,0.596,0.908
4,Other,310,0.116,0.094,0.667,0.018,0.828,0.738,0.967


[Married-civ-spouse 집단 내 성별 비교]


,group,n,actual_pos_rate,pred_pos_rate,TPR(recall),FPR,precision,F1,AUC
0,Male,15983,0.444,0.512,0.804,0.279,0.696,0.746,0.856
1,Female,1967,0.450,0.542,0.837,0.301,0.694,0.759,0.857


## 9. 결론 (요약은 README의 '추가 분석' 절에 정리)

- 같은 피처·같은 분할에서 **GBDT 계열이 선형 모델·RF보다 꾸준히 높고**, 제출 CatBoost가 그중 최고였다 → CatBoost 선택의 근거가 숫자로 확보됐다.
- 피처 블록별 효과는 모델에 따라 달라 **보고서의 "cross-row 통계만 기여" 주장은 CatBoost 기준 ablation 수치로 뒷받침해야 한다** (이 노트북을 catboost 설치 환경에서 다시 실행하면 자동으로 CatBoost 기준 표가 나온다).
- Target Encoding을 split 전에 계산하면 작지만 체계적인 낙관 편향이 생긴다 → 다음 버전에서는 **split-first / fold 내 인코딩**이 기본이어야 한다.
- 단일 holdout 수치는 fold 간 표준편차만큼 흔들린다 → **CV 평균±표준편차**로 보고하는 편이 정직하다.
- 공정성: 예측 양성률 격차는 실제 양성률 격차를 거의 그대로 반영하고, 혼인상태를 통제하면 성별 간 TPR/FPR 차이가 크게 줄어든다 → 모델이 성별을 직접 쓰기보다 혼인상태·가구관계를 통해 간접 반영한다. 다만 소수 인종 집단은 표본이 적어 AUC 추정이 불안정하다.